# 01 — Exploratory Data Analysis
## "Your Zip Code is Your Health Sentence"

This notebook loads the master dataset (5 merged federal sources), explores distributions, correlations, and geographic patterns to guide the statistical analysis in `02_analysis.ipynb`.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.loaders import load_data, describe_data, FIPS_COL
from src.loaders.health_data import download_all
from src.loaders.merge import build_master
from src.viz.plots import distribution_grid, correlation_matrix, category_counts
from src.analysis.stats import summary_stats

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 6)})
%matplotlib inline

DATA_PROCESSED = Path("..") / "data" / "processed"

In [ ]:
# Build or load the master dataset
master_path = DATA_PROCESSED / "master.parquet"
if master_path.exists():
    df = pd.read_parquet(master_path)
    print(f"Loaded master: {df.shape[0]:,} rows x {df.shape[1]} cols")
else:
    print("Master not found — building from scratch (downloads ~5 datasets)...")
    df = build_master(save=True)
    print(f"Built master: {df.shape[0]:,} rows x {df.shape[1]} cols")

## Data Overview

## Health Outcome Distributions

## Food Desert vs Non-Food-Desert Health Comparison

## Correlation Matrix — Health + Socioeconomic Variables

## Scatter: Food Access Score vs Diabetes Rate (by Majority Race)

In [ ]:
# Correlation heatmap of key variables
corr_cols = [
    "diabetes_pct", "obesity_pct", "life_expectancy",
    "is_food_desert", "poverty_rate", "median_household_income",
    "pct_black", "pct_hispanic", "pct_bachelors_plus",
    "uninsured_pct", "physical_inactivity_pct",
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix: Health Outcomes × Socioeconomic Factors", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: poverty rate vs diabetes, colored by food desert status
if "poverty_rate" in df.columns and "diabetes_pct" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: poverty vs diabetes, colored by food desert
    scatter_data = df.dropna(subset=["poverty_rate", "diabetes_pct", "is_food_desert"]).sample(
        min(5000, len(df)), random_state=42
    )
    colors = {0: "#2ecc71", 1: "#e74c3c"}
    labels = {0: "Non-Food-Desert", 1: "Food Desert"}
    for fd in [0, 1]:
        subset = scatter_data[scatter_data["is_food_desert"] == fd]
        axes[0].scatter(subset["poverty_rate"], subset["diabetes_pct"],
                       alpha=0.15, s=8, c=colors[fd], label=labels[fd])
    axes[0].set_xlabel("Poverty Rate (%)")
    axes[0].set_ylabel("Diabetes Prevalence (%)")
    axes[0].set_title("Poverty Rate vs Diabetes by Food Desert Status")
    axes[0].legend()

    # Right: poverty vs diabetes, colored by majority race
    if "majority_race" in df.columns:
        race_colors = {"White": "#3498db", "Black": "#e74c3c", "Hispanic": "#f39c12"}
        for race, color in race_colors.items():
            subset = scatter_data[scatter_data["majority_race"] == race]
            if len(subset) > 0:
                axes[1].scatter(subset["poverty_rate"], subset["diabetes_pct"],
                               alpha=0.15, s=8, c=color, label=race)
        axes[1].set_xlabel("Poverty Rate (%)")
        axes[1].set_ylabel("Diabetes Prevalence (%)")
        axes[1].set_title("Poverty Rate vs Diabetes by Majority Race")
        axes[1].legend()

    plt.tight_layout()
    plt.show()

## Life Expectancy by Income Quintile

In [ ]:
# Life expectancy by income quintile, split by majority race
if "life_expectancy" in df.columns and "income_quintile" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: box plot by income quintile
    le_data = df.dropna(subset=["life_expectancy", "income_quintile"])
    sns.boxplot(data=le_data, x="income_quintile", y="life_expectancy", ax=axes[0], palette="RdYlGn")
    axes[0].set_xlabel("Income Quintile (1=poorest, 5=richest)")
    axes[0].set_ylabel("Life Expectancy (years)")
    axes[0].set_title("Life Expectancy by Income Quintile")

    q_means = le_data.groupby("income_quintile")["life_expectancy"].mean()
    if len(q_means) >= 2:
        gap = q_means.iloc[-1] - q_means.iloc[0]
        axes[0].text(0.5, 0.02, f"Gap (Q5-Q1): {gap:.1f} years",
                    transform=axes[0].transAxes, fontsize=11, ha="center",
                    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

    # Right: by income quintile AND race
    if "majority_race" in df.columns:
        le_race = df.dropna(subset=["life_expectancy", "income_quintile", "majority_race"])
        race_q = le_race.groupby(["income_quintile", "majority_race"])["life_expectancy"].mean().unstack()
        race_q.plot(kind="bar", ax=axes[1], color=["#e74c3c", "#f39c12", "#3498db"], width=0.8)
        axes[1].set_xlabel("Income Quintile")
        axes[1].set_ylabel("Mean Life Expectancy")
        axes[1].set_title("Life Expectancy by Income × Race")
        axes[1].legend(title="Majority Race")
        axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    plt.show()

## Income × Race → Diabetes Heatmap

In [ ]:
# Income quintile x majority race → diabetes prevalence heatmap
if all(c in df.columns for c in ["income_quintile", "majority_race", "diabetes_pct"]):
    matrix = df.dropna(subset=["income_quintile", "majority_race", "diabetes_pct"]).pivot_table(
        index="income_quintile", columns="majority_race",
        values="diabetes_pct", aggfunc="mean",
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
                linewidths=1, cbar_kws={"label": "Mean Diabetes %"})
    ax.set_xlabel("Majority Race of Census Tract")
    ax.set_ylabel("Income Quintile (1=poorest, 5=richest)")
    ax.set_title("Diabetes Prevalence: Income Quintile × Majority Race")
    plt.tight_layout()
    plt.show()

    print("\nKey observation: Compare high-income Black tracts vs low-income White tracts")
    if "Black" in matrix.columns and "White" in matrix.columns:
        hi_black = matrix.loc[matrix.index.max(), "Black"] if matrix.index.max() in matrix.index else None
        lo_white = matrix.loc[matrix.index.min(), "White"] if matrix.index.min() in matrix.index else None
        if hi_black and lo_white:
            print(f"  Richest Black tracts: {hi_black:.1f}% diabetes")
            print(f"  Poorest White tracts: {lo_white:.1f}% diabetes")
            print(f"  → Income {'does NOT close' if hi_black > lo_white else 'closes'} the racial gap")

In [ ]:
# --- CORRELATIONS ---
# correlation_matrix(df)

In [ ]:
# --- CATEGORY EXPLORATION ---
# category_counts(df, "column_name")

In [ ]:
# --- EXTENDED STATS ---
# summary_stats(df)